In [1]:
!pip install selenium

In [12]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
import time
import pandas as pd

driver = webdriver.Chrome()

def scrapeMovieData(driver):
    movie_names_list = []
    movie_duration_list = []
    movie_ratings_list = []
    user_voting_count_list = []

    while True:

        try:
            button=driver.find_element(By.XPATH,"//button[@class='ipc-btn ipc-btn--single-padding ipc-btn--center-align-content ipc-btn--default-height ipc-btn--core-base ipc-btn--theme-base ipc-btn--button-radius ipc-btn--on-accent2 ipc-text-button ipc-see-more__button']")
            button.send_keys(Keys.ENTER)
        except NoSuchElementException:
            print("Reached the end of scrolling")
            
            movies_containers = driver.find_elements(By.XPATH,".//div[@class='sc-dc48a950-0 kaBWmw']")
            break
        except:
            print("Something else went wrong") 
            break
        else:
            time.sleep(5)

    def movieNameFormat(movie_name):
        name = movie_name.split(" ")
        name[0] = ''
        final_movie_name = " ".join(name)
        return final_movie_name.lstrip()
    
    def movieDurationToMinutes(durations):
        total_minutes = 0
        for duration in durations:
            duration = str(duration.text)
            if duration.endswith('m') or duration.endswith('h'):
                total_duration = duration.split()
                if total_duration[0].endswith('h'):
                    total_duration[0] = total_duration[0].replace('h','')
                    total_minutes = int(total_duration[0])*60
                elif total_duration[0].endswith('m'):
                    total_duration[0] = total_duration[0].replace('m','')
                    total_minutes = int(total_duration[0])

                if len(total_duration) > 1:
                    total_duration[1] = total_duration[1].replace('m','')
                    total_minutes = total_minutes + int(total_duration[1])

        return total_minutes

    def totalVotingCount(voting_count):
        final_voting_count = 0.0
        voting_count = voting_count.replace('(', '').replace(')', '')
        if voting_count.endswith('K'):
            voting_count = voting_count.replace('K','')
            final_voting_count = float(voting_count) * 1000
        elif voting_count.endswith('M'):
            voting_count = voting_count.replace('M','')
            final_voting_count = float(voting_count) * 1000000
        else:
            final_voting_count = float(voting_count)

        return int(final_voting_count)

    for container in movies_containers:

        movie_name = container.find_elements(By.XPATH,".//h3[@class='ipc-title__text ipc-title__text--reduced']")
        movie_names_list.append(movieNameFormat(str(movie_name[0].text)))

        durations = container.find_elements(By.XPATH,".//span[@class='sc-dc48a950-8 gikOtO dli-title-metadata-item']")
        movie_duration_list.append(movieDurationToMinutes(durations))

        ratings = container.find_elements(By.XPATH,".//span[@class='ipc-rating-star--rating']")

        if len(ratings) != 0:
            movie_ratings_list.append(float(ratings[0].text))
        else:
            movie_ratings_list.append("0")

        voting_count = container.find_elements(By.XPATH,".//span[@class='ipc-rating-star--voteCount']")
        
        if len(voting_count) != 0:
            user_voting_count_list.append(totalVotingCount(str(voting_count[0].text)))
        else:
            user_voting_count_list.append("0")
        

    genre = driver.find_elements(By.XPATH,"//span[@class='ipc-chip__text']")
    n = len(movie_names_list)
    genre_list = [genre[2].text] * n

    df = pd.DataFrame()
    df['Movie Name'] = movie_names_list
    df['Movie Genre'] = genre_list
    df['Movie Duration'] = movie_duration_list
    df['Movie Rating'] = movie_ratings_list
    df['User Votes'] = user_voting_count_list
    return df
    
movie_genres = ['animation','adventure','sci-fi', 'reality-tv','talk-show']

imdb_movies_df = pd.DataFrame()
imdb_movies_df['Movie Name'] = []
imdb_movies_df['Movie Genre'] = []
imdb_movies_df['Movie Duration'] = []
imdb_movies_df['Movie Rating'] = []
imdb_movies_df['User Votes'] = []

for genre in movie_genres:
    driver.get(f'https://www.imdb.com/search/title/?title_type=feature&release_date=2024-01-01,2024-12-31&genres={genre}')
    time.sleep(5)
    imdb_movies_df = pd.concat([imdb_movies_df, scrapeMovieData(driver)])

imdb_movies_df.to_csv(r"/Users/dheivasubramanian/Downloads/IMDB PROJECT_UNO/DATA/IMDB_MOVIE_DATA.csv", index= False)



Reached the end of scrolling
Reached the end of scrolling
Reached the end of scrolling
Reached the end of scrolling
Reached the end of scrolling


In [ ]:
import mysql.connector

connection = mysql.connector.connect(
    host = "localhost",
    user = "root",
    password = "Roronoa7*",
    database = "IMDB_Dataset"
)
cursor = connection.cursor()

#query = "create database IMDB_Dataset"
query = "create table imdb_movie_data(movie_name varchar(200), genre varchar(50), duration int, rationgs float, user_vote_count int)" 

cursor.execute(query)



In [7]:
import pandas as pd
csv_df = pd.read_csv(r'/Users/dheivasubramanian/Downloads/IMDB PROJECT_UNO/DATA/IMDB_MOVIE_DATA.csv')

data = []

for i in csv_df.index:
    row = csv_df.loc[i].values
    row[0] = str(row[0])
    row[1] = str(row[1])
    row[2] = int(row[2])
    row[3] = float(row[3])
    row[4] = int(row[4])

    data.append(tuple(row))
query = "insert into imdb_movie_data values(%s,%s,%s,%s,%s)"
cursor.executemany(query,data)
connection.commit()



In [11]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 36.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.2/731.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.8/30.8 MB 35.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [streamlit]14 [streamlit]


In [12]:
csv_df

,Movie Name,Movie Genre,Movie Duration,Movie Rating,User Votes
0,The Wild Robot,Animation,102.0,8.2,178000
1,Moana 2,Animation,100.0,6.6,108000
2,Flow,Animation,85.0,7.9,83000
3,Mufasa: The Lion King,Animation,118.0,6.6,70000
4,Paddington in Peru,Animation,106.0,6.7,25000
...,...,...,...,...,...
1689,Night of Recovery: Live from the Waynesboro Th...,Talk-Show,121.0,0.0,0
1690,The Poetry of Clark L. Stanton,Talk-Show,47.0,0.0,0
1691,Darkman Audio Commentary with Josh Ruben,Talk-Show,0.0,0.0,0
1692,Paul Molinar: Supercut,Talk-Show,129.0,0.0,0
